# 05 · Análisis de errores, encodings cruzados y lo que no funcionó

El modelo del `04_modelos_ensamble.ipynb` (texto v4 + promedio LightGBM/XGBoost) llega a **0,9433 en la validación** y **0,94466 en Kaggle**. En el 04 se vio que cambiar de algoritmo casi no mueve la nota, así que este notebook vuelve a las **features**:

1. **Análisis de errores:** dónde y por qué se equivoca el modelo.
2. **Ideas para los vendedores con pocas publicaciones**, que es donde más se equivoca: perfil del vendedor con predicciones, vecinos por título, suavizado por vendedor.
3. **Target encodings cruzados** (vendedor × tipo de publicación, etc.). **Es lo único que funcionó.**
4. **Señales escondidas** en los datos: orden de las filas, IDs, formato de las fotos, lotes de publicaciones, títulos editados.
5. **Un modelo lineal** que ve texto y vendedor juntos.
6. **Hiperparámetros** con las features nuevas.
7. **Modelo final y submission.**

**Cómo se evalúa cada idea:** igual que en el 04, contra una referencia y **partición por partición**. Una idea se acepta solo si mejora el promedio más que el ruido (~0,001) y además mejora en **casi todas las particiones**. Para filtrar rápido, las ideas se prueban con **LightGBM solo**, porque XGBoost tarda el doble y en el 04 se vio que los dos se comportan igual. Lo que pasa el filtro se confirma con el modelo completo.

Todo el notebook tarda unas **2 horas**. Que la compu no se suspenda.

## 0. Datos, features y piezas

Lo mismo que en el 04, pero con el texto v4 (`caracteres=True`), que es el del modelo actual. Tarda unos 12 minutos.

In [1]:
import re, time, unicodedata
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from features import (cargar_datos, armar_tabla, calcular_piezas, armar_conjunto, texto_combinado,
                      target_encoding, prob_texto, claves_cruzadas, g)

todo, y = cargar_datos()
N_TRAIN = len(y)
IDX_TEST = np.arange(N_TRAIN, len(todo))
X_todo = armar_tabla(todo)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
PARTICIONES = list(cv.split(np.zeros(N_TRAIN), y)) + [(np.arange(N_TRAIN), IDX_TEST)]

inicio = time.time()
piezas = [calcular_piezas(todo, y, idx_a, idx_b, caracteres=True) for idx_a, idx_b in PARTICIONES]
print(X_todo.shape, f"piezas listas en {(time.time() - inicio) / 60:.1f} minutos")

(100000, 92) piezas listas en 8.4 minutos


Funciones para evaluar:

- `evaluar_lgbm(extra)`: LightGBM con las features de siempre + las piezas de `piezas[f]` + lo que haya en `extra`. `extra[f]` es un diccionario `nombre -> (valores de entrenamiento, valores de validación)` para la partición `f`. Devuelve la probabilidad de usado de cada ítem de train (*out of fold*).
- `columna(valores)`: convierte una columna calculada sobre `todo` (sin usar la respuesta) al formato de `extra`.
- `te(clave)`: *target encoding* de una clave nueva, calculado adentro de cada partición (fuera de fold), en el formato de `extra`.
- `comparar(nombre, p, ref)`: accuracy y diferencia **partición por partición** contra una referencia.

In [2]:
PARAMS_LGBM = dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, colsample_bytree=0.5, random_state=42, verbose=-1)
PARAMS_XGB  = dict(n_estimators=1500, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.5,
                   min_child_weight=2, tree_method="hist", enable_categorical=True, random_state=42, n_jobs=8)
sub = lambda s, idx: s.iloc[idx].reset_index(drop=True)

def evaluar_lgbm(extra=None, params=PARAMS_LGBM, modelo=LGBMClassifier):
    p = np.zeros(N_TRAIN)
    for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
        A, B = armar_conjunto(X_todo, {**piezas[f], **(extra[f] if extra else {})}, idx_a, idx_b)
        p[idx_b] = modelo(**params).fit(A, y[idx_a]).predict_proba(B)[:, 1]
    return p

def columna(nombre, valores):
    valores = np.asarray(valores)
    return [{nombre: (valores[idx_a], valores[idx_b])} for idx_a, idx_b in PARTICIONES[:5]]

def te(nombre, clave):
    return [{nombre: target_encoding(sub(clave, idx_a), y[idx_a], sub(clave, idx_b))} for idx_a, idx_b in PARTICIONES[:5]]

def juntar(*extras):
    return [{k: v for e in extras for k, v in e[f].items()} for f in range(5)]

def accuracy_por_particion(p):
    return np.array([accuracy_score(y[idx_b], p[idx_b] > 0.5) for _, idx_b in PARTICIONES[:5]])

resultados = []
def comparar(nombre, p, ref):
    notas, notas_ref = accuracy_por_particion(p), accuracy_por_particion(ref)
    dif = notas - notas_ref
    resultados.append({"experimento": nombre, "accuracy": round(notas.mean(), 4), "vs. referencia": round(dif.mean(), 4),
                       "mejora en": f"{(dif > 0).sum()}/5", "diferencia por partición": " ".join(f"{d:+.4f}" for d in dif)})
    print(f"{nombre:55s} {notas.mean():.4f}  ({dif.mean():+.4f}, mejora en {(dif > 0).sum()}/5)")

Referencias: el **modelo 04 completo** (LightGBM + XGBoost) y **LightGBM solo** con las mismas features, que es contra lo que se filtran las ideas. Tarda unos 10 minutos.

In [3]:
p_lgbm_04 = evaluar_lgbm()
p_xgb_04  = evaluar_lgbm(params=PARAMS_XGB, modelo=XGBClassifier)
p_04 = (p_lgbm_04 + p_xgb_04) / 2
print(f"modelo 04 (LightGBM + XGBoost): {accuracy_por_particion(p_04).mean():.4f}")
print(f"LightGBM solo:                  {accuracy_por_particion(p_lgbm_04).mean():.4f}")

modelo 04 (LightGBM + XGBoost): 0.9433
LightGBM solo:                  0.9431


## 1. Análisis de errores

### 1.1 Según el tamaño del vendedor

Ya en el 04 se vio que el modelo se equivoca mucho más con vendedores que tienen pocas publicaciones (contando train + test):

In [4]:
errores = (p_04 > 0.5) != y
n_items = X_todo["vendedor_n_items"].iloc[:N_TRAIN].values
grupos = pd.cut(n_items, [0, 1, 5, 50, np.inf], labels=["1 ítem", "2 a 5", "6 a 50", "más de 50"])
tabla = pd.DataFrame({"grupo": grupos, "error": errores}).groupby("grupo", observed=True)["error"].agg(["mean", "size", "sum"])
tabla.columns = ["tasa de error", "ítems", "errores"]
tabla["% de todos los errores"] = tabla["errores"] / tabla["errores"].sum()
tabla.round(3)

,tasa de error,ítems,errores,% de todos los errores
grupo,,,,
1 ítem,0.110,17503,1925,0.485
2 a 5,0.056,15611,877,0.221
6 a 50,0.037,24341,908,0.229
más de 50,0.020,12545,256,0.065


### 1.2 Los errores "seguros"

Errores donde el modelo estaba **muy seguro** (probabilidad menor a 0,15 o mayor a 0,85) y aun así se equivocó:

In [5]:
tr = todo.iloc[:N_TRAIN]
casos = pd.DataFrame({"título": tr["title"].values, "real": np.where(y == 1, "used", "new"), "p_usado": p_04.round(2),
                      "tipo": tr["listing_type_id"].values, "ítems del vendedor": n_items,
                      "garantía": tr["warranty"].fillna("").str[:35].values})
mal = casos[errores]
seguros = mal[(mal["p_usado"] < 0.15) | (mal["p_usado"] > 0.85)]
print(f"errores: {len(mal)}, de los cuales 'seguros': {len(seguros)} ({len(seguros) / len(mal):.0%})")
print("\nREAL USADO, el modelo estaba seguro de NUEVO:")
display(seguros[seguros["real"] == "used"].sort_values("p_usado").head(12))
print("REAL NUEVO, el modelo estaba seguro de USADO:")
display(seguros[seguros["real"] == "new"].sort_values("p_usado", ascending=False).head(12))

errores: 3966, de los cuales 'seguros': 1395 (35%)

REAL USADO, el modelo estaba seguro de NUEVO:


,título,real,p_usado,tipo,ítems del vendedor,garantía
147,Bisel Marco Frontal De Carcasa Lg G Pro E980 E...,used,0.0,bronze,108,si
363,Intercambiadores Para Calderas Varios Modelos ...,used,0.0,bronze,1,3 meses
62545,Recarga Toner Xerox Xc 3220 - 3210 -cod 106r01487,used,0.0,bronze,2,
61771,"Matafuegos Polvo 2,5 Kg Paternal Sello Iram Ca...",used,0.0,gold_special,1,Con garantía de 1 año para la carga
50574,Marco Para Cuadro O Espejo - Oferta!!!,used,0.0,bronze,21,Sin garantía
50884,Volver A Empezar Paola C. Alvarez ...,used,0.0,bronze,10,100% DE CALIFICACIONES POSITIVAS +
63021,Alarma A2k4 - Alonso Hnos,used,0.0,bronze,1,DEL FABRICANTE
4169,Mercedes Benz Travego Bus 1:43 Minichamps Unico!,used,0.0,bronze,19,Sin garantía
18641,100 Soldaditos De Guerra Plasticos Souvenirs J...,used,0.0,bronze,26,
44499,Platos Tsuji 450 Ala Corta Gastronomicos 19 21...,used,0.0,silver,50,


REAL NUEVO, el modelo estaba seguro de USADO:


,título,real,p_usado,tipo,ítems del vendedor,garantía
45862,"Revista Ciencia, Docencia Y Tecnología N° 21 (...",new,1.0,bronze,43,
40162,"Preparacion Para El Choque Pachter, Henry (ar...",new,1.0,bronze,134,Sí
50840,Rolando Laserie Urgencia Simple Argentino Promo,new,1.0,bronze,293,Sin garantía
50192,Hablemos De Niños - D.gonzalez Torres (a5-psi),new,1.0,bronze,101,Sin garantía
43699,Fisica General Tomo 3. Ramon Loyarte. Calor,new,1.0,bronze,117,Sin garantía
6874,Medio Equipo Silens Corsa 3 Puertas (excelente),new,1.0,silver,1,
38626,Sabor De Hombre De Sofia Klahr,new,1.0,bronze,172,Sí
36802,Antigua Guia Practica Mar Del Plata Temporada ...,new,1.0,bronze,16,mis calificaciones
10419,Historia Continental De Iberoamerica - 3 Tomos...,new,1.0,bronze,8,
10365,El Régimen Alimenticio En Las Enfermedades Del...,new,1.0,bronze,65,Sin garantía


**Lectura:** un tercio de los errores son "seguros", y muchos son **etiquetas que contradicen la propia publicación**: libros de texto con "Usado" en el título marcados como nuevos, antigüedades y vinilos de los 70 marcados como nuevos, ebooks (que no pueden ser usados) marcados como usados, cables "( Nuevos )" marcados como usados.

La condición la **declara el vendedor** al publicar, y muchos la cargan mal o con su propio criterio. Ningún modelo puede acertar esos casos. Es **ruido en la etiqueta**, y pone un techo a la accuracy alcanzable.

El margen de mejora está en los errores **dudosos** (probabilidad entre 0,15 y 0,85) y en los vendedores con pocas publicaciones. A eso apuntan las ideas que siguen.

## 2. Ideas para vendedores con pocas publicaciones

### 2.1 Perfil del vendedor con predicciones del texto

El *target encoding* del vendedor solo usa **train**, porque necesita la respuesta. Pero muchos vendedores tienen publicaciones **en test** o en la partición de validación, cuya respuesta no se conoce. La idea es promediar lo que el **modelo de texto** opina de las **otras** publicaciones del vendedor (train + validación + test), sin contar la propia.

**Para que no haya *leakage*:** en cada partición, el modelo de texto se entrena **solo con las 4 particiones de entrenamiento** y con él se predicen la de validación **y test**. Así, ninguna respuesta de la partición de validación entra en el cálculo. Tarda unos 12 minutos.

In [6]:
texto = texto_combinado(todo)
vendedor = todo["seller_id"].values

inicio = time.time()
texto_test = [prob_texto(sub(texto, idx_a), y[idx_a], sub(texto, IDX_TEST), caracteres=True)[1]
              for idx_a, _ in PARTICIONES[:5]]
print(f"listo en {(time.time() - inicio) / 60:.1f} minutos")

def promedio_otros(idx_pool, p_pool, idx_obj):
    """Para cada fila de idx_obj: promedio de p sobre las OTRAS filas del mismo vendedor que están en el pool."""
    d = pd.DataFrame({"v": vendedor[idx_pool], "p": p_pool})
    suma, cant = d.groupby("v")["p"].sum(), d.groupby("v")["p"].size()
    v_obj = pd.Series(vendedor[idx_obj])
    s, c = v_obj.map(suma).fillna(0).values, v_obj.map(cant).fillna(0).values
    propia = pd.Series(idx_obj).map(pd.Series(p_pool, index=idx_pool)).values
    esta = ~np.isnan(propia)
    s, c = s - np.where(esta, propia, 0), c - esta
    return np.where(c > 0, s / np.maximum(c, 1), np.nan), c

perfil = []
for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
    pa, pb = piezas[f]["prob_combinado"]
    idx_pool = np.concatenate([idx_a, idx_b, IDX_TEST])
    p_pool = np.concatenate([pa, pb, texto_test[f]])
    (ma, ca), (mb, cb) = promedio_otros(idx_pool, p_pool, idx_a), promedio_otros(idx_pool, p_pool, idx_b)
    perfil.append({"vend_prob_texto_otros": (ma, mb), "vend_n_otros": (ca, cb)})

comparar("2.1 perfil del vendedor con predicciones del texto", evaluar_lgbm(perfil), p_lgbm_04)

listo en 6.7 minutos


2.1 perfil del vendedor con predicciones del texto      0.9428  (-0.0003, mejora en 1/5)


### 2.2 Vecinos por título

Para un vendedor con **una sola publicación** no hay perfil posible. Pero si vende "Iphone 5s 16gb Libre", otros vendedores publicaron cosas parecidas. Para cada ítem se buscan los **10 títulos más parecidos** (similitud coseno sobre TF-IDF) de **otros vendedores**, y se mira si eran nuevos o usados:

- `knn_media`: proporción de usados entre los 10 vecinos, ponderada por parecido.
- `knn_sim_max`: qué tan parecido es el vecino más cercano.
- `knn_top1`: si el vecino más cercano era usado.
- `knn_n_sim05`: cuántos vecinos tienen similitud mayor a 0,5.

Se excluye al **mismo vendedor** (si no, sería repetir el *target encoding*). Contra el *leakage*, en cada partición los vecinos de las filas de entrenamiento se buscan **fuera de fold** (en las otras 4 sub-particiones), y los de validación, en todo el entrenamiento. Tarda unos 5 minutos.

In [7]:
titulos = todo["title"].fillna("").values
K = 10

def vecinos(idx_ref, idx_obj):
    vect = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, dtype=np.float32)
    R, O = vect.fit_transform(titulos[idx_ref]), vect.transform(titulos[idx_obj])
    RT, y_ref, v_ref = R.T.tocsr(), y[idx_ref], vendedor[idx_ref]
    salida = np.zeros((len(idx_obj), 4), np.float32)
    for i in range(0, len(idx_obj), 1000):   # de a 1000 filas para no llenar la memoria
        sl = slice(i, i + 1000)
        sim = (O[sl] @ RT).toarray()
        sim[vendedor[idx_obj[sl]][:, None] == v_ref[None, :]] = 0           # fuera el mismo vendedor
        top = np.argpartition(-sim, K, axis=1)[:, :K]
        s, yy = np.take_along_axis(sim, top, 1), y_ref[top]
        orden = np.argsort(-s, axis=1)
        s, yy = np.take_along_axis(s, orden, 1), np.take_along_axis(yy, orden, 1)
        salida[sl, 0] = (s * yy).sum(1) / np.maximum(s.sum(1), 1e-6)
        salida[sl, 1], salida[sl, 2], salida[sl, 3] = s[:, 0], yy[:, 0], (s > 0.5).sum(1)
    salida[salida[:, 1] == 0, 0] = np.nan
    return salida

NOMBRES_KNN = ["knn_media", "knn_sim_max", "knn_top1", "knn_n_sim05"]
inicio = time.time()
knn = []
for idx_a, idx_b in PARTICIONES[:5]:
    fa = np.zeros((len(idx_a), 4), np.float32)
    for a2, b2 in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(idx_a, y[idx_a]):
        fa[b2] = vecinos(idx_a[a2], idx_a[b2])
    fb = vecinos(idx_a, idx_b)
    knn.append({n: (fa[:, j], fb[:, j]) for j, n in enumerate(NOMBRES_KNN)})
print(f"listo en {(time.time() - inicio) / 60:.1f} minutos")

comparar("2.2 vecinos por título", evaluar_lgbm(knn), p_lgbm_04)

listo en 2.9 minutos


2.2 vecinos por título                                  0.9433  (+0.0003, mejora en 3/5)


**Lectura:** ninguna de las dos mejora más que el ruido.

- El **perfil con predicciones** repite lo que ya dicen el *target encoding* y el perfil del vendedor del 03. Y los vendedores que más lo necesitan, los de una sola publicación, no tienen "otras publicaciones" para promediar.
- Los **vecinos por título** tampoco: el texto combinado ya captura casi todo lo que aportan los vecinos.

## 3. Target encodings cruzados

El *target encoding* del vendedor resume **todo** su historial en un número. Pero un mismo vendedor puede tener **varios "modos"**: un comercio que vende repuestos nuevos en publicaciones *gold* y liquida cosas usadas en publicaciones *free*, o alguien que vende libros usados y celulares nuevos. Se prueba el *target encoding* de **combinaciones**:

- **vendedor × tipo de publicación** (`listing_type_id`: gold, silver, bronze, free...).
- **vendedor × categoría.**
- **categoría × primera palabra del título**: una especie de "tipo de producto" ("libro", "remera", "iphone" dentro de su categoría). Sirve también para vendedores sin historial.

Mismo cálculo que siempre: suavizado 10, fuera de fold en entrenamiento, adentro de cada partición.

In [8]:
CRUZADAS = claves_cruzadas(todo)
te_cruzados = {n: te(n, clave) for n, clave in CRUZADAS.items()}

comparar("3.1 + vendedor x categoría",          evaluar_lgbm(te_cruzados["te_vend_cat"]), p_lgbm_04)
comparar("3.2 + vendedor x tipo de publicación", evaluar_lgbm(te_cruzados["te_vend_tipo"]), p_lgbm_04)
comparar("3.3 + categoría x primera palabra",   evaluar_lgbm(te_cruzados["te_cat_palabra1"]), p_lgbm_04)
comparar("3.4 + tipo + palabra (sin vend x cat)",
         evaluar_lgbm(juntar(te_cruzados["te_vend_tipo"], te_cruzados["te_cat_palabra1"])), p_lgbm_04)
TRES_TE = juntar(*te_cruzados.values())
p_lgbm_3te = evaluar_lgbm(TRES_TE)
comparar("3.5 + los 3 juntos", p_lgbm_3te, p_lgbm_04)

3.1 + vendedor x categoría                              0.9430  (-0.0001, mejora en 1/5)


3.2 + vendedor x tipo de publicación                    0.9439  (+0.0008, mejora en 4/5)


3.3 + categoría x primera palabra                       0.9432  (+0.0002, mejora en 3/5)


3.4 + tipo + palabra (sin vend x cat)                   0.9442  (+0.0011, mejora en 5/5)


3.5 + los 3 juntos                                      0.9444  (+0.0013, mejora en 5/5)


**Lectura:**

- **Vendedor × tipo de publicación** es el que más suma solo, en línea con la idea de los "modos" del vendedor.
- Los otros dos solos no suman, pero **los tres juntos dan la mayor mejora y mejoran en las 5 particiones**. Se complementan: cada uno corta el historial del vendedor de una manera distinta.
- Sin vendedor × categoría (3.4) queda apenas por debajo de los tres juntos (3.5); la diferencia es ruido, pero los tres juntos dan el mejor resultado y mejoran en 5/5, así que se quedan **los tres**.

### 3.6 ¿Otros cruces o suavizados?

Si funcionan estos, puede haber más. Se prueba agregar otros cruces **sobre los 3 elegidos** (referencia: LightGBM con los 3), y cambiar el suavizado del vendedor (3 y 30 en vez de 10). Tarda unos 20 minutos.

In [9]:
vend, cat = todo["seller_id"].astype(str), todo["category_id"].astype(str)
titulo_min = todo["title"].fillna("").str.lower()
palabra1 = titulo_min.str.extract(r"([a-záéíóúñ]+)", expand=False).fillna("")
OTROS_CRUCES = {
    "te_vend_envio":     vend + "|" + todo["shipping"].apply(lambda s: str(g(s, "mode"))),
    "te_vend_garantia":  vend + "|" + todo["warranty"].notna().astype(str),
    "te_vend_cuadrada":  vend + "|" + X_todo["fotos_cuadradas"].fillna(-1).gt(0.5).astype(str),
    "te_vend_precio":    vend + "|" + pd.qcut(np.log1p(todo["price"]), 10, labels=False, duplicates="drop").astype(str),
    "te_vend_cant1":     vend + "|" + (todo["initial_quantity"] == 1).astype(str),
    "te_cat_tipo":       cat + "|" + todo["listing_type_id"],
    "te_cat_palabras2":  cat + "|" + titulo_min.str.extract(r"([a-záéíóúñ]+\W+[a-záéíóúñ0-9]+)", expand=False).fillna(palabra1),
}
for n, clave in OTROS_CRUCES.items():
    comparar(f"3.6 + {n}", evaluar_lgbm(juntar(TRES_TE, te(n, clave))), p_lgbm_3te)

for s in [3, 30]:
    otro = [{"te_seller_id": target_encoding(sub(todo["seller_id"], ia), y[ia], sub(todo["seller_id"], ib), suavizado=s)}
            for ia, ib in PARTICIONES[:5]]
    comparar(f"3.6 suavizado del vendedor = {s}", evaluar_lgbm(juntar(TRES_TE, otro)), p_lgbm_3te)

3.6 + te_vend_envio                                     0.9439  (-0.0005, mejora en 1/5)


3.6 + te_vend_garantia                                  0.9443  (-0.0000, mejora en 4/5)


3.6 + te_vend_cuadrada                                  0.9443  (-0.0001, mejora en 2/5)


3.6 + te_vend_precio                                    0.9442  (-0.0002, mejora en 3/5)


3.6 + te_vend_cant1                                     0.9443  (-0.0000, mejora en 3/5)


3.6 + te_cat_tipo                                       0.9439  (-0.0005, mejora en 1/5)


3.6 + te_cat_palabras2                                  0.9443  (-0.0001, mejora en 3/5)


3.6 suavizado del vendedor = 3                          0.9441  (-0.0003, mejora en 1/5)


3.6 suavizado del vendedor = 30                         0.9442  (-0.0001, mejora en 3/5)


**Lectura:** ningún cruce más suma sobre los tres elegidos, y el suavizado 10 sigue siendo el mejor. La información de "modos" del vendedor ya queda cubierta con los tres.

## 4. Señales escondidas en los datos

Algunos competidores están por encima, así que vale la pena buscar señales que no se miraron. Primero, una exploración rápida:

In [10]:
def proporcion_usados(nombre, valores):
    t = pd.DataFrame({"valor": np.asarray(valores), "usado": y}).groupby("valor", observed=True)["usado"].agg(["mean", "size"])
    t.columns = ["proporción de usados", "ítems"]
    print(f"--- {nombre}"); display(t.sort_values("ítems", ascending=False).head(8).round(3))

# orden de las filas
print("¿la fila tiene la misma respuesta que la anterior?", round((y[1:] == y[:-1]).mean(), 3),
      "| esperable por azar:", round(y.mean() ** 2 + (1 - y.mean()) ** 2, 3))
# número del id del ítem
proporcion_usados("id del ítem (deciles)", pd.qcut(tr["id"].str[3:].astype(np.int64), 10, labels=False))
# formato del id de las fotos
ids_fotos = todo["pictures"].apply(lambda L: [p["id"] for p in L])
proporcion_usados("largo del prefijo del id de la primera foto", ids_fotos.iloc[:N_TRAIN].apply(lambda L: len(L[0].split("-")[0]) if L else 0))
# descripción que pertenece a otro ítem
desc_de_otro = np.array([int(len(d) > 0 and i not in str(d)) for i, d in zip(todo["id"], todo["descriptions"])])
proporcion_usados("la descripción es de otro ítem", desc_de_otro[:N_TRAIN])
# título editado: el link conserva el título original
def slug(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "-", t).strip("-")
link = todo["permalink"].fillna("").str.extract(r"MLA\d+-(.*)-_JM", expand=False).fillna("")
titulo_editado = np.array([int(a != slug(b)[:len(a)]) for a, b in zip(link, todo["title"])])
proporcion_usados("título editado (distinto del link)", titulo_editado[:N_TRAIN])
# lotes: publicaciones del mismo vendedor el mismo día
fecha = pd.to_datetime(todo["date_created"], utc=True)
dia, semana = fecha.dt.floor("D").astype(str), fecha.dt.strftime("%G-%V")
lote_dia_n = todo.groupby([todo["seller_id"], dia])["id"].transform("size").values
proporcion_usados("publicaciones del vendedor ese mismo día", np.minimum(lote_dia_n[:N_TRAIN], 10))

¿la fila tiene la misma respuesta que la anterior? 0.503 | esperable por azar: 0.503


--- id del ítem (deciles)


,proporción de usados,ítems
valor,,
0,0.463,7000
1,0.454,7000
2,0.473,7000
3,0.463,7000
4,0.464,7000
5,0.472,7000
6,0.472,7000
7,0.454,7000


--- largo del prefijo del id de la primera foto


,proporción de usados,ítems
valor,,
6,0.427,35920
5,0.462,21678
4,0.563,11519
0,0.680,541
3,0.537,285
2,0.551,49
1,0.625,8


...

,proporción de usados,ítems
valor,,
0,0.463,70000


--- título editado (distinto del link)


,proporción de usados,ítems
valor,,
0,0.480,57886
1,0.383,12114


--- publicaciones del vendedor ese mismo día


,proporción de usados,ítems
valor,,
1,0.437,38946
2,0.446,9211
10,0.468,8897
3,0.504,4368
4,0.552,2609
5,0.586,1857
6,0.597,1386
7,0.565,1127


**Lo que se ve:**

- **Orden de las filas** e **id del ítem**: al azar, sin señal (los IDs están anonimizados).
- **Formato del id de la foto**: los prefijos cortos son del sistema viejo de fotos de MercadoLibre y tienen más usados. Seguramente se solapa con la antigüedad de la foto del 03.
- **Descripción de otro ítem** (publicación copiada o relanzada) y **título editado**: algo de señal.
- **Lotes**: los vendedores que suben muchas cosas el mismo día tienen más usados, y dentro de un lote la condición es más homogénea que en todo el vendedor.

Se prueban sobre los 3 encodings cruzados:

In [11]:
X_fotos = {
    "foto_largo_prefijo": ids_fotos.apply(lambda L: np.mean([len(i.split("-")[0]) for i in L]) if L else np.nan),
    "foto_sufijo_viejo":  ids_fotos.apply(lambda L: np.mean([len(i.split("_")[-1]) != 6 for i in L]) if L else np.nan),
}
X_fotos["vend_prop_foto_vieja"] = X_fotos["foto_sufijo_viejo"].groupby(todo["seller_id"]).transform("mean")
ids_y_desc = juntar(*[columna(n, v) for n, v in X_fotos.items()], columna("desc_de_otro_item", desc_de_otro))
comparar("4.1 + formato de ids de fotos y descripción", evaluar_lgbm(juntar(TRES_TE, ids_y_desc)), p_lgbm_3te)

te_lote = juntar(te("te_vend_dia", vend + "|" + dia), te("te_vend_semana", vend + "|" + semana))
comparar("4.2 + target encoding vendedor x día y x semana", evaluar_lgbm(juntar(TRES_TE, te_lote)), p_lgbm_3te)
lote_semana_n = todo.groupby([todo["seller_id"], semana])["id"].transform("size").values
tam_lote = juntar(columna("lote_dia_n", lote_dia_n), columna("lote_semana_n", lote_semana_n))
comparar("4.3 + tamaño del lote", evaluar_lgbm(juntar(TRES_TE, tam_lote)), p_lgbm_3te)
comparar("4.4 + título editado", evaluar_lgbm(juntar(TRES_TE, columna("titulo_editado", titulo_editado))), p_lgbm_3te)

4.1 + formato de ids de fotos y descripción             0.9443  (-0.0001, mejora en 2/5)


4.2 + target encoding vendedor x día y x semana         0.9442  (-0.0002, mejora en 3/5)


4.3 + tamaño del lote                                   0.9439  (-0.0005, mejora en 1/5)


4.4 + título editado                                    0.9436  (-0.0008, mejora en 0/5)


**Lectura:** nada de esto mejora más que el ruido. Las señales existen, pero ya estaban cubiertas: la antigüedad de la foto (03) cubre el formato del id, y los encodings cruzados del vendedor cubren los lotes.

## 5. Suavizado por vendedor

Variante de la idea 2.1, ahora con las predicciones del **modelo completo** (mucho más fuertes que las del texto solo) y aplicada **al final**, no como feature:

`p_final = (1 − α) · p_ítem + α · promedio de las predicciones de las otras publicaciones del vendedor sin respuesta (validación + test)`

Para eso, en cada partición el modelo entrenado con las 4 particiones de entrenamiento predice validación **y test** juntos. Se prueba aplicarlo a todos los ítems, o solo a los de vendedores **sin historial en entrenamiento** (los que el *target encoding* no conoce). Tarda unos 3 minutos.

In [12]:
claves_te = {"te_seller_id": todo["seller_id"], "te_category_id": todo["category_id"], **CRUZADAS}
p_val, p_tst = np.zeros(N_TRAIN), []
for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
    idx_bt = np.concatenate([idx_b, IDX_TEST])
    pz = {n: target_encoding(sub(c, idx_a), y[idx_a], sub(c, idx_bt)) for n, c in claves_te.items()}
    pa, pb = piezas[f]["prob_combinado"]
    pz["prob_combinado"] = (pa, np.concatenate([pb, texto_test[f]]))
    orden = ["te_seller_id", "te_category_id", "prob_combinado", *CRUZADAS]
    A, B = armar_conjunto(X_todo, {n: pz[n] for n in orden}, idx_a, idx_bt)
    p = LGBMClassifier(**PARAMS_LGBM).fit(A, y[idx_a]).predict_proba(B)[:, 1]
    p_val[idx_b] = p[:len(idx_b)]
    p_tst.append(p[len(idx_b):])
print("sin suavizar (igual a 3.5):", accuracy_por_particion(p_val).mean().round(4))

for alfa in [0.1, 0.2, 0.3, 0.4]:
    for solo_sin_historial in [False, True]:
        q = p_val.copy()
        for f, (idx_a, idx_b) in enumerate(PARTICIONES[:5]):
            media, cant = promedio_otros(np.concatenate([idx_b, IDX_TEST]), np.concatenate([p_val[idx_b], p_tst[f]]), idx_b)
            historial = pd.Series(vendedor[idx_a]).value_counts().reindex(vendedor[idx_b]).fillna(0).values
            usar = (cant > 0) & ((historial == 0) if solo_sin_historial else True)
            q[idx_b] = np.where(usar, (1 - alfa) * p_val[idx_b] + alfa * np.nan_to_num(media), p_val[idx_b])
        comparar(f"5. suavizado alfa={alfa}" + (" (solo sin historial)" if solo_sin_historial else ""), q, p_lgbm_3te)

sin suavizar (igual a 3.5): 0.9444
5. suavizado alfa=0.1                                   0.9442  (-0.0002, mejora en 1/5)


5. suavizado alfa=0.1 (solo sin historial)              0.9443  (-0.0001, mejora en 1/5)
5. suavizado alfa=0.2                                   0.9440  (-0.0004, mejora en 1/5)


5. suavizado alfa=0.2 (solo sin historial)              0.9443  (-0.0000, mejora en 2/5)
5. suavizado alfa=0.3                                   0.9437  (-0.0006, mejora en 0/5)


5. suavizado alfa=0.3 (solo sin historial)              0.9444  (+0.0000, mejora en 2/5)
5. suavizado alfa=0.4                                   0.9429  (-0.0015, mejora en 0/5)


5. suavizado alfa=0.4 (solo sin historial)              0.9444  (+0.0000, mejora en 2/5)


**Lectura:** no mejora. Aplicado a todos, cuanto más peso, peor: mezcla al ítem con publicaciones distintas del mismo vendedor. Aplicado solo a vendedores sin historial, queda igual. Las publicaciones sin respuesta del vendedor no agregan nada que el modelo no sepa.

## 6. Modelo lineal: texto y vendedor juntos

Hoy el texto y el vendedor entran **por separado**: el texto como probabilidad, el vendedor como *target encoding*. Una **regresión logística** que vea **a la vez** las palabras (TF-IDF de palabras + caracteres) y una columna 0/1 por cada vendedor, categoría, tipo de publicación, vendedor × tipo, tamaño de foto, rango de precio, etc., puede aprender el efecto del vendedor **más allá** de lo que dicen sus palabras.

Los vectorizadores y el *one-hot* se ajustan sobre train + test (no usan la respuesta). La logística sí la usa, así que va fuera de fold como siempre. Se prueba de dos formas: como **feature** de LightGBM y **promediada** con LightGBM. Tarda unos 8 minutos.

In [13]:
inicio = time.time()
W = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, dtype=np.float32).fit_transform(texto)
C_ = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, sublinear_tf=True, max_features=200_000,
                     dtype=np.float32).fit_transform(texto)
categoricas = pd.DataFrame({
    "vend": vend, "cat": cat, "tipo": todo["listing_type_id"], "vend_tipo": vend + todo["listing_type_id"], "vend_cat": vend + cat,
    "modo": todo["buying_mode"], "envio": todo["shipping"].apply(lambda s: str(g(s, "mode"))),
    "foto": todo["pictures"].apply(lambda L: L[0]["max_size"] if L else "no"),
    "cant": todo["initial_quantity"].clip(upper=10).astype(str),
    "precio": pd.qcut(np.log1p(todo["price"]), 20, labels=False, duplicates="drop").astype(str)})
M = sp.hstack([W, C_, OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit_transform(categoricas)]).tocsr()
print("matriz:", M.shape)

def prob_lineal(idx_a, idx_b):
    pa = np.zeros(len(idx_a))
    for a2, b2 in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(idx_a, y[idx_a]):
        pa[b2] = LogisticRegression(max_iter=3000, C=1).fit(M[idx_a[a2]], y[idx_a[a2]]).predict_proba(M[idx_a[b2]])[:, 1]
    return pa, LogisticRegression(max_iter=3000, C=1).fit(M[idx_a], y[idx_a]).predict_proba(M[idx_b])[:, 1]

lineal = [{"prob_lineal": prob_lineal(idx_a, idx_b)} for idx_a, idx_b in PARTICIONES[:5]]
p_lineal = np.zeros(N_TRAIN)
for f, (_, idx_b) in enumerate(PARTICIONES[:5]):
    p_lineal[idx_b] = lineal[f]["prob_lineal"][1]
print(f"listo en {(time.time() - inicio) / 60:.1f} minutos | modelo lineal solo: {accuracy_por_particion(p_lineal).mean():.4f}")

comparar("6.1 + modelo lineal como feature", evaluar_lgbm(juntar(TRES_TE, lineal)), p_lgbm_3te)
for w in [0.1, 0.2, 0.3]:
    comparar(f"6.2 promedio con el lineal (peso {w})", (1 - w) * p_lgbm_3te + w * p_lineal, p_lgbm_3te)

matriz: (100000, 415061)


listo en 7.7 minutos | modelo lineal solo: 0.9239


6.1 + modelo lineal como feature                        0.9446  (+0.0003, mejora en 4/5)
6.2 promedio con el lineal (peso 0.1)                   0.9445  (+0.0001, mejora en 3/5)
6.2 promedio con el lineal (peso 0.2)                   0.9440  (-0.0004, mejora en 0/5)
6.2 promedio con el lineal (peso 0.3)                   0.9439  (-0.0004, mejora en 2/5)


**Lectura:** el modelo lineal **solo** es sorprendentemente bueno (~0,924), pero no suma. Como feature, lo que sabe ya lo tiene LightGBM. Promediado, es bastante más débil que LightGBM y empeora la nota. **Se descarta.**

## 7. Hiperparámetros con las features nuevas

Con 3 features nuevas, puede que convenga otra configuración. Se prueban variantes alrededor de la actual (referencia: LightGBM con los 3 encodings). Tarda unos 15 minutos.

In [14]:
grilla = [
    dict(num_leaves=31, n_estimators=2500),
    dict(num_leaves=127, n_estimators=1000),
    dict(colsample_bytree=0.3, n_estimators=2500),
    dict(min_child_samples=10),
    dict(learning_rate=0.01, n_estimators=3000),
    dict(subsample=0.8, subsample_freq=1),
]
for cambio in grilla:
    comparar("7. " + ", ".join(f"{k}={v}" for k, v in cambio.items()), evaluar_lgbm(TRES_TE, params={**PARAMS_LGBM, **cambio}), p_lgbm_3te)

7. num_leaves=31, n_estimators=2500                     0.9439  (-0.0004, mejora en 2/5)


7. num_leaves=127, n_estimators=1000                    0.9440  (-0.0004, mejora en 2/5)


7. colsample_bytree=0.3, n_estimators=2500              0.9441  (-0.0003, mejora en 2/5)


7. min_child_samples=10                                 0.9441  (-0.0003, mejora en 2/5)


7. learning_rate=0.01, n_estimators=3000                0.9443  (-0.0000, mejora en 3/5)


7. subsample=0.8, subsample_freq=1                      0.9440  (-0.0004, mejora en 1/5)


**Lectura:** ninguna variante le gana a la configuración actual. Se mantiene.

## 8. Resumen y modelo final

Lo único que pasó el filtro son los **3 encodings cruzados**. Se confirma con el modelo completo (LightGBM + XGBoost) contra el modelo 04:

In [15]:
p_xgb_3te = evaluar_lgbm(TRES_TE, params=PARAMS_XGB, modelo=XGBClassifier)
p_05 = (p_lgbm_3te + p_xgb_3te) / 2
comparar("8. MODELO 05: 04 + 3 encodings cruzados (LightGBM + XGBoost)", p_05, p_04)
comparar("8. LightGBM solo con los 3 encodings", p_lgbm_3te, p_04)
pd.DataFrame(resultados)

8. MODELO 05: 04 + 3 encodings cruzados (LightGBM + XGBoost) 0.9443  (+0.0010, mejora en 5/5)
8. LightGBM solo con los 3 encodings                    0.9444  (+0.0010, mejora en 5/5)


,experimento,accuracy,vs. referencia,mejora en,diferencia por partición
0,2.1 perfil del vendedor con predicciones del t...,0.9428,-0.0003,1/5,-0.0004 +0.0011 -0.0007 -0.0009 -0.0004
1,2.2 vecinos por título,0.9433,0.0003,3/5,+0.0006 +0.0012 -0.0006 -0.0002 +0.0004
2,3.1 + vendedor x categoría,0.9430,-0.0001,1/5,+0.0000 +0.0005 -0.0004 -0.0004 -0.0001
3,3.2 + vendedor x tipo de publicación,0.9439,0.0008,4/5,-0.0001 +0.0017 +0.0004 +0.0006 +0.0016
4,3.3 + categoría x primera palabra,0.9432,0.0002,3/5,-0.0001 +0.0009 -0.0013 +0.0004 +0.0011
5,3.4 + tipo + palabra (sin vend x cat),0.9442,0.0011,5/5,+0.0016 +0.0020 +0.0002 +0.0003 +0.0015
6,3.5 + los 3 juntos,0.9444,0.0013,5/5,+0.0016 +0.0016 +0.0010 +0.0010 +0.0014
7,3.6 + te_vend_envio,0.9439,-0.0005,1/5,-0.0011 -0.0003 -0.0016 -0.0001 +0.0009
8,3.6 + te_vend_garantia,0.9443,-0.0000,4/5,+0.0001 +0.0005 -0.0010 +0.0001 +0.0001
9,3.6 + te_vend_cuadrada,0.9443,-0.0001,2/5,-0.0004 +0.0000 -0.0014 +0.0004 +0.0007


**Lectura:** el modelo 05 mejora al 04 en las 5 particiones. Con los encodings cruzados, LightGBM solo y el promedio con XGBoost dan prácticamente lo mismo (la diferencia es menor que el ruido). Se mantiene el promedio, para cambiar **una sola cosa** respecto del 04 (las features) y porque promediar dos modelos reduce la variabilidad.

**Submission:** igual que el 04, pero las piezas se calculan con `cruzados=True` (los 3 encodings, con todo train → test).

In [16]:
import os

idx_a, idx_b = PARTICIONES[5]
piezas_final = calcular_piezas(todo, y, idx_a, idx_b, caracteres=True, cruzados=True)
A, B = armar_conjunto(X_todo, piezas_final, idx_a, idx_b)
p_test = (LGBMClassifier(**PARAMS_LGBM).fit(A, y).predict_proba(B)[:, 1]
          + XGBClassifier(**PARAMS_XGB).fit(A, y).predict_proba(B)[:, 1]) / 2

sub05 = pd.DataFrame({"ID": range(1, len(B) + 1), "condition": np.where(p_test > 0.5, "used", "new")})
os.makedirs("submissions", exist_ok=True)
sub05.to_csv("submissions/05_encodings_cruzados.csv", index=False)

anterior = pd.read_csv("submissions/04_lgbm_xgb_texto_v4.csv")
print(A.shape, sub05["condition"].value_counts(normalize=True).round(3).to_dict())
print(f"cambian {(sub05['condition'] != anterior['condition']).sum()} predicciones respecto de la submission 04")

(70000, 98) {'new': 0.529, 'used': 0.471}
cambian 265 predicciones respecto de la submission 04
